# Getting Started

**The FunGen-xQTL protocol is a reproducible, end-to-end pipeline for molecular quantitative trait loci (QTL) analysis** — from raw genotypes and phenotypes through discovery, fine-mapping, and integration with GWAS.

This page is a guided on-ramp. A minimal toy dataset of **49 de-identified samples** is used throughout the examples so you can try every pipeline end-to-end before running on real data. In about an hour you'll install the environment, clone the repo, download the demo dataset, and run your first cis-QTL scan.

```{image} images/complete_workflow.png
:alt: FunGen-xQTL analysis workflow
:align: center
:width: 90%
```

:::{seealso}
**New to the consortium?** Start with [How to use the resource](https://statfungen.github.io/xqtl-protocol/README.html#how-to-use-the-resource) on the homepage for the big-picture background, then come back here to set up.
:::


---

## At a Glance

The protocol is modular. Each numbered pipeline is a self-contained [SoS (Script of Scripts)](https://vatlab.github.io/sos-docs/) notebook that can run independently or be chained into the full workflow.

| Stage | What it does | Key pipelines |
|---|---|---|
| **1. Preprocess** | Clean, normalize, and align inputs | phenotype QC, genotype QC, covariate generation |
| **2. Discover** | Scan for QTLs | TensorQTL (cis/trans), APEX (interactions) |
| **3. Fine-map** | Identify credible causal variants | SuSiE, mvSuSiE, fSuSiE |
| **4. Integrate** | Link QTLs to disease and biology | coloc, cTWAS, GWAS integration, enrichment |

Full details with links to every mini-protocol are further down in [Analysis](#analysis). For now, let's get you set up.


---

## Before You Start

You'll need a Linux or macOS shell. Windows users: install [WSL2](https://learn.microsoft.com/windows/wsl/install) first, then follow the Linux path.

| Requirement | Minimum | Recommended |
|---|---|---|
| Disk space | 10 GB (minimal install) | 40 GB (full bioinformatics stack) |
| Memory | 16 GB | 50 GB+ on HPC for the installer |
| Network | GitHub, conda-forge, synapse.org | Same |
| Git | Any recent version | 2.30+ |

:::{tip}
**On HPC** — make sure you have access to a compute node with at least 50 GB of memory for the pixi installation step (Step 2). Login nodes often kill large installs. See Step 2 for details.
:::


---

## Step 1. Install SoS in a Conda Environment

The protocol's pipelines are written as [SoS (Script of Scripts)](https://vatlab.github.io/sos-docs/) workflows. First, create a dedicated conda environment and install SoS along with its language modules. Full installation reference: [SoS Conda installation guide](https://vatlab.github.io/sos-docs/running.html#Conda-installation).

If you don't have conda yet, install [Miniforge](https://github.com/conda-forge/miniforge) (recommended) or [Anaconda](https://www.anaconda.com/download).

```bash
# Create and activate a new environment for SoS
conda create -n sos python=3.12 -y
conda activate sos

# Install the full SoS suite
conda install -c conda-forge \
    sos sos-pbs sos-notebook jupyterlab-sos sos-papermill \
    sos-bash sos-python sos-r

# Register the SoS kernel with Jupyter
python -m sos_notebook.install
```

**Verify:**

```bash
sos --version
jupyter kernelspec list    # should include 'sos'
```

:::{tip}
Make sure you always `conda activate sos` before running any pipeline commands.
:::


---

## Step 2. Install the xQTL Software Stack with pixi

Next, install the bioinformatics and data-science packages the protocol depends on using [pixi](https://pixi.sh/) via the [StatFunGen/pixi-setup](https://github.com/StatFunGen/pixi-setup) installer.

**On HPC systems**, your home directory likely has a storage quota that won't fit the full install. Temporarily point `$HOME` to a path with enough space, and add pixi to your `$PATH`:

```bash
# Point HOME to a location with enough disk space
export HOME="/your_pixi_install_path"

# Add pixi to your path
export PATH="/your_pixi_install_path/.pixi/bin:$PATH"
```

Then run the installer:

```bash
curl -fsSL https://raw.githubusercontent.com/StatFunGen/pixi-setup/refs/heads/main/pixi-setup.sh | bash
```

**On a laptop or workstation** you can skip the `HOME`/`PATH` exports and just run the `curl` command — the installer will prompt you to choose an install path and type interactively.

The installer will prompt you for two things:

**1. Installation path** — where pixi stores environments and packages.

| Setting | When to use |
|---|---|
| `$HOME/.pixi` (default) | Laptops and workstations with plenty of home-directory space |
| `/your_pixi_install_path/.pixi` | HPC systems with strict home-directory quotas |

**2. Installation type**

| Type | Size | Files | Includes |
|---|---|---|---|
| **1. minimal** | ~5 GB | ~100k | CLI tools, Python data-science stack, JupyterLab, base R (tidyverse, devtools, IRkernel) |
| **2. full** | ~35 GB | ~350k | Everything above, **plus** samtools, bcftools, plink2, GATK4, STAR, Seurat, tensorQTL, Bioconductor |

Choose **minimal** for xQTL runs with pre-processed inputs; choose **full** if you'll also do upstream QC, alignment, or single-cell work.

**Activate and verify:**

```bash
source ~/.bashrc          # or ~/.zshrc on macOS
pixi --version
```

You should see a version number. If not, open a fresh terminal.

:::{warning}
**On HPC**, run the installer from a compute node with at least 50 GB of memory, not the login node. The install process can be memory-intensive and may be killed on login nodes:

```bash
srun --mem=50G --pty bash          # SLURM
bsub -Is -M 50000 -n 4 bash        # LSF
```
:::


---

## Step 3. Clone the Protocol

```bash
git clone https://github.com/StatFunGen/xqtl-protocol.git
cd xqtl-protocol
```

:::{note}
**What's in the repo**

| Folder | Contents |
|---|---|
| `pipeline/` | The SoS workflows you'll run |
| `code/` | Notebook-based documentation (this page lives here) |
| `data/` | Small example inputs and configuration templates |
| `website/` | JupyterBook sources for [statfungen.github.io/xqtl-protocol](https://statfungen.github.io/xqtl-protocol/) |
:::


---

## Step 4. Download the Demo Data

Preparation of the demo dataset is documented [on this page](https://github.com/cumc/fungen-xqtl-analysis/tree/main/analysis/Wang_Columbia/ROSMAP/MWE) (a private repository accessible to FunGen-xQTL working group members). The data itself lives on [Synapse](https://www.synapse.org/#!Synapse:syn36416601).

1. Create a free account on [synapse.org](https://www.synapse.org/) — username and password are required to download.
2. Install the Synapse API client into pixi's python environment:
   ```bash
   pixi global install -c conda-forge --environment python synapseclient
   ```
   Alternatively, `pip install synapseclient`. See [the Synapse install docs](https://help.synapse.org/docs/Installing-Synapse-API-Clients.1985249668.html) for details.
3. Every folder at each level of the Synapse project has its own unique ID, so you can download just the subset you need.

**Bulk RNA-seq molecular phenotype quantification** — test data:

```python
import synapseclient
import synapseutils
syn = synapseclient.Synapse()
syn.login("your username on synapse.org", "your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn53174239', path="./")
```

**xQTL association analysis** — test data:

```python
import synapseclient
import synapseutils
syn = synapseclient.Synapse()
syn.login("your username on synapse.org", "your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn52369482', path="./")
```


---

## Step 5. Run Your First Workflow

Confirm SoS can see the pipelines:

```bash
sos run pipeline/1_xqtl_association.ipynb -h
```

You should see a list of workflow options. Now run a minimal cis-QTL scan using the demo data you just downloaded:

```bash
sos run pipeline/TensorQTL.ipynb cis \
    --genotype-file   data/example/genotype.bed \
    --phenotype-file  data/example/phenotype.bed.gz \
    --covariate-file  data/example/covariates.tsv \
    --cwd output/demo_tensorqtl
```

Results land in `output/demo_tensorqtl/`. You now have a working environment and a known-good reference run to compare against when you bring in your own data.

:::{tip}
Every pipeline supports `-h` and `--help`, and SoS prints the exact shell commands it runs under the hood — a great way to learn what's happening and to debug failures.
:::


## Analysis

With the environment set up, here's the full protocol in order. Each link is a self-contained mini-protocol; all commands in them should be executed from the command line with `sos run pipeline/<name>.ipynb ...`.

:::{important}
**Minimum Working Example (MWE) — new users, start here.**

Every module in the repo ships a minimal `MWE`-prefixed test dataset under [Synapse `syn36416559`](https://www.synapse.org/#!Synapse:syn36416559/files/). To go end-to-end on the demo data, run these **five** pipelines in order and skip everything else on the first pass:

1. [`reference_data.ipynb`](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data.html) — pull the standardized reference files
2. [`bulk_expression.ipynb`](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/bulk_expression.html) — quantify gene expression (MWE default)
3. [`genotype_preprocessing.ipynb`](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/genotype_preprocessing.html) → [`phenotype_preprocessing.ipynb`](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype_preprocessing.html) → [`covariate_preprocessing.ipynb`](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/covariate_preprocessing.html) — QC + normalization
4. [`qtl_association_testing.ipynb`](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_testing.html) — run cis-QTL with TensorQTL
5. [`mnm_miniprotocol.ipynb`](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_miniprotocol.html) — single-trait fine-mapping + TWAS with SuSiE

Once this pass completes cleanly, branch out to the additional modules below (methylation, splicing, multivariate mixture, GWAS integration, enrichment, EMS) based on what your project needs.
:::

### 1. Reference Data

Before quantifying phenotypes, set up the standardized reference files — genomes, gene annotations, variant annotations, LD maps, and topologically associated domains.

- [Reference data setup](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data.html) — main entry point ⭐ *MWE*
- [Reference data preparation](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data_preparation.html) — detailed preparation steps
- [Generalized TAD-B](https://statfungen.github.io/xqtl-protocol/code/reference_data/generalized_TADB.html) — TAD boundaries for analysis windows
- [LD reference pruning](https://statfungen.github.io/xqtl-protocol/code/reference_data/ld_prune_reference.html) and [RSS LD sketching](https://statfungen.github.io/xqtl-protocol/code/reference_data/rss_ld_sketch.html) — advanced LD utilities

### 2. Molecular Phenotypes

We support bulk RNA-seq, DNA methylation, and alternative splicing phenotypes. Each path has its own calling, QC, and normalization steps.

- **Bulk RNA-seq** — [bulk_expression mini-protocol](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/bulk_expression.html) ⭐ *MWE*, with sub-modules for [RNA calling](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/RNA_calling.html), [QC](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/bulk_expression_QC.html), and [normalization](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/bulk_expression_normalization.html)
- **DNA methylation** — [methylation mini-protocol](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/methylation.html) with [methylation calling via SeSAMe](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/methylation_calling.html)
- **Alternative splicing** — [splicing mini-protocol](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/splicing.html) with [splicing calling via leafcutter2](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/splicing_calling.html) and [normalization](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/splicing_normalization.html)

### 3. Data Pre-processing

- [Genotype preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/genotype_preprocessing.html) ⭐ *MWE* — VCF QC, GWAS QC, PCA, GRM, plink formatting
- [Phenotype preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype_preprocessing.html) ⭐ *MWE* — gene annotation, imputation, formatting
- [Covariate preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/covariate_preprocessing.html) ⭐ *MWE* — merge genetic PCs with phenotypes, compute hidden factors

### 4. QTL Association Testing

- [QTL association testing](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_testing.html) ⭐ *MWE* — [TensorQTL](https://statfungen.github.io/xqtl-protocol/code/association_scan/TensorQTL/TensorQTL.html) scans (cis, trans, interaction) and [quantile regression QTL](https://statfungen.github.io/xqtl-protocol/code/association_scan/quantile_models/qr_and_twas.html)
- [Association postprocessing](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_postprocessing.html) — hierarchical multiple testing and p-value adjustment

### 5. Multivariate Mixture Model

Learn a data-driven mixture prior across contexts/tissues for multivariate fine-mapping.

- [Multivariate mixture vignette](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/multivariate_mixture_vignette.html) — overview
- [Mixture prior with MASH](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mixture_prior.html) and [MASH fit](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_fit.html) — data-driven prior estimation

### 6. Multiomics Regression Models

Fine-mapping and multi-context regression — the core of the post-discovery analysis.

- [Multi-omic regression mini-protocol](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_miniprotocol.html) ⭐ *MWE* — start here
- [Univariate fine-mapping + TWAS with SuSiE](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_twas_vignette.html)
- [Multivariate multi-gene fine-mapping](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/multivariate_multigene_fine_mapping_vignette.html)
- [Univariate fine-mapping with fSuSiE](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_fsusie_vignette.html) — functional / epigenomic data
- [Multivariate fine-mapping vignette](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/multivariate_fine_mapping_vignette.html)
- [Summary-statistics fine-mapping](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/summary_stats_finemapping_vignette.html)
- [Multi-omic multi-trait regression](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/mnm_regression.html) and [RSS analysis](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/rss_analysis.html)
- [MNM postprocessing](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_postprocessing.html)

### 7. GWAS Integration

Link xQTL signals to disease-associated loci.

- [SuSiE-enloc colocalization](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/SuSiE_enloc.html) — pairwise colocalization of xQTL and GWAS fine-mapping
- [TWAS / cTWAS](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html) — causal TWAS for complex traits
- [Colocboost](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/colocboost.html) — shared-variant discovery across multiple molecular traits

### 8. Enrichment and Validation

- [Excess-of-overlap enrichment](https://statfungen.github.io/xqtl-protocol/code/enrichment/eoo_enrichment.html) — significance of variants in annotation sets
- [Pathway enrichment (GSEA)](https://statfungen.github.io/xqtl-protocol/code/enrichment/gsea.html)
- [GREGOR](https://statfungen.github.io/xqtl-protocol/code/enrichment/gregor.html) — annotation-based enrichment for significant variants
- [Stratified LD Score Regression](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html) — heritability partitioning by annotation

### 9. xQTL Modifier Score (EMS)

Train and apply a per-variant score for prioritizing regulatory variants.

- [EMS training](https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_training.html)
- [EMS prediction](https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_prediction.html)

### Command Generator (shortcut)

Want to skip writing SoS commands by hand? The [eQTL analysis command generator](https://statfungen.github.io/xqtl-protocol/code/commands_generator/eQTL_analysis_commands.html) produces the full pipeline from a single configuration file — great for reproducing a run or sharing a recipe.


---

## Software Environment

Every protocol on this site runs inside the pixi environment configured in Steps 1–2. Once pixi and SoS are installed, each example "just works" — no per-pipeline container, no manual dependency wrangling.

Need something extra? Install it into the right pixi environment:

```bash
# Python package (into the shared python env)
pixi global install -c conda-forge --environment python <package>

# R package (into the r-base env)
pixi global install -c conda-forge --environment r-base r-<package>

# Standalone bioinformatics CLI tool
pixi global install -c bioconda <tool>
```

### Troubleshooting

:::{warning}
**R library conflicts.** If you see an error like

```
Error in dyn.load(file, DLLpath = DLLPath, ...):
unable to load shared object '$PATH/R/x86_64-pc-linux-gnu-library/4.2/stringi/libs/stringi.so':
libicui18n.so.63: cannot open shared object file: No such file or directory
```

your system R libraries are being picked up alongside the pixi ones. Unset them before running the pipeline:

```bash
export R_LIBS=""
export R_LIBS_USER=""
```
:::

**`pixi: command not found`** — open a new terminal, or re-source your shell rc file (`source ~/.bashrc` on Linux/HPC, `source ~/.zshrc` on macOS).

**Installer killed on HPC** — you're on a login node. Request a compute node with ≥ 50 GB memory and re-run.

**`sos: command not found`** — Step 2 didn't complete. Re-run the `pixi global install` command for SoS.

**`ModuleNotFoundError` during a pipeline** — install the missing package into pixi's python env with the command above.

Still stuck? [Open an issue](https://github.com/StatFunGen/xqtl-protocol/issues) with the command you ran and the full error output.


---

## Analyses on High Performance Computing Clusters

The demo on this page runs on a desktop workstation. Production analyses typically run on an HPC cluster, and SoS supports this natively via [SoS Remote Tasks](https://vatlab.github.io/sos-docs/doc/user_guide/task_statement.html) on [configured host computers](https://vatlab.github.io/sos-docs/doc/user_guide/host_setup.html).

We provide a [toy example for running SoS pipelines on a typical HPC cluster environment](https://github.com/statfungen/xqtl-protocol/blob/main/code/misc/Job_Example.ipynb) — first-time users are encouraged to work through it before launching real jobs. It covers the host and task configuration you'll reuse for every subsequent pipeline, and it's schedule-agnostic (SLURM, LSF, SGE, PBS/Torque all work).
